In [ ]:
import os, json
import pandas as pd

from src.distribution_calibration import DistributionCalibration, DistributionCalibrationBatch

In [ ]:
data_dir = os.path.join("..", "..", "data")
movie_lens_data_dir = os.path.join(data_dir, "MovieLens-20M", "distribution_calibration")
dataset_name = "MovieLens"

# Read and prepare data

In [ ]:
top_movies_df = pd.read_csv(os.path.join(movie_lens_data_dir, "top_movies_df.csv"))
persona_ratings = pd.read_csv(os.path.join(movie_lens_data_dir, "persona_ratings.csv"), index_col=0) # index is movie id

# Ratings grid
possible_ratings = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
possible_ratings_str = [f"{r:.1f}" for r in possible_ratings]

# Create dataframe with rating counts
dataset_counts = top_movies_df[possible_ratings_str]

# Convert to probabilities by dividing each row by its sum
P_df = dataset_counts.div(dataset_counts.sum(axis=1), axis=0)
P = P_df.values

# Get digital twin ratings
persona_cols = [c for c in persona_ratings.columns if c.startswith('pid_')]
Y_hat = persona_ratings[persona_cols].values

# Load personas to examine top personas
with open(os.path.join(data_dir, 'personas.json'), 'r') as f:
    personas = json.load(f)

# get the list of pids in persona_ratings
pids = persona_cols

# One run

In [ ]:
# Create optimizer
opt = DistributionCalibration(
    P, 
    Y_hat, 
    dataset_name,
    personas,
    pids,
    possible_ratings, 
    divergence='l1', 
    method='mirror_descent', 
    reg_w=1e-6, 
    reg_v=1e-6, 
    reg_mse=1e-7, 
    fit_persona_only=False, 
    fit_dummy_only=False, 
    weight_tol=None,
    max_iter=100000, 
    tol=1e-5, 
    learning_rate=1e-2, 
    train_test_ratio=0.8, 
    random_state=42,
    adaptive_lr=True, 
    max_grad_norm=10.0
)

# Run full workflow
results = opt.run_full_workflow(
    plot_convergence=True,
    plot_variance_ratio=True,
    plot_dist_difference=True,
    test_question_idx=0,
    get_top_weights=True,
    top_num=10,
    plot_divergence_comparison=True
)

# Multiple runs

In [ ]:
opt_batch = DistributionCalibrationBatch(
    P,
    Y_hat,
    dataset_name,
    personas,
    pids,
    possible_ratings,
    divergences=['tv', 'chi2', 'kl', 'hellinger', 'ks', 'l1', 'l2'],
    method='mirror_descent',
    reg_w=1e-6,
    reg_v=1e-6,
    reg_mse=1e-7,
    weight_tol=None,
    max_iter=100000,
    tol=1e-5,
    learning_rate=1e-2,
    train_test_ratio=0.8,
    random_state=42,
    adaptive_lr = True,
    max_grad_norm = 10.0,
)

results = opt_batch.run_all_experiments()